# Q-Learning Case Study

Notebook version of `q_learning_case_study.py`.

## Imports And Policy Helpers

Import NumPy, random seeding utilities, and the shared gridworld. The helper functions below convert a Q-table into an epsilon-greedy policy and sample actions from that policy during exploration.


In [1]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np

from gridworld_case_study import ACTION_NAMES, GridWorldCaseStudyEnv, format_policy


def epsilon_greedy_policy_from_q(Q, epsilon):
    """Construct an epsilon-greedy behavior policy from a Q-table.

    Precondition:
    `Q` is a two-dimensional state-action value table and `epsilon` is a valid
    exploration rate between 0 and 1.

    What happens:
    1. Start each state's action distribution with uniform exploration mass.
    2. Find the single `argmax` action in each row.
    3. Add the remaining `1 - epsilon` probability to that greedy action.

    Postcondition:
    Returns a policy matrix whose rows are valid action distributions for use by
    the sampling and learning routines in this notebook.
    """
    num_states, num_actions = Q.shape
    policy = np.full((num_states, num_actions), epsilon / num_actions)

    for s in range(num_states):
        best_action = int(np.argmax(Q[s]))
        # ensuring that the best action gets the remaining probability mass, ensuring that the row sums to 1.0
        policy[s, best_action] += 1.0 - epsilon

    return policy


def sample_action_from_policy(policy, state):
    """Sample one action index from the policy row for the given state.

    Precondition:
    `policy[state]` is a valid categorical distribution over actions.

    What happens:
    1. Select the requested state's probability row.
    2. Draw one action index using NumPy's categorical sampling.

    Postcondition:
    Returns the sampled action as an integer suitable for `env.step()`.
    """
    return int(np.random.choice(policy.shape[1], p=policy[state]))


## Q-Learning Loop

The next section performs temporal-difference control. For each episode it interacts with the environment, applies the Q-learning update using the maximum next-state action value, and gradually decays epsilon so the behavior policy shifts from exploration toward exploitation.


In [2]:
def q_learning(
    env,
    num_episodes=5000,
    alpha=0.1,
    gamma=0.9,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    max_steps_per_episode=50,
    seed=7,
):
    """Train a Q-table with the standard off-policy Q-learning update.

    Precondition:
    `env` exposes a finite discrete action/state interface, and the learning
    hyperparameters describe a sensible training setup.

    What happens:
    1. Seed Python and NumPy randomness for reproducibility.
    2. Initialize the Q-table to zeros.
    3. For each episode, roll out an epsilon-greedy behavior policy.
    4. Apply the Q-learning target `r + gamma * max_a' Q(s', a')` after each
       transition.
    5. Decay epsilon after each episode while respecting `epsilon_min`.

    Postcondition:
    Returns the learned Q-table and the final epsilon value after training.
    """
    random.seed(seed)
    np.random.seed(seed)
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    Q = np.zeros((num_states, num_actions))

    for _ in range(num_episodes):
        state, _ = env.reset()
        done = False

        for _ in range(max_steps_per_episode):
            policy = epsilon_greedy_policy_from_q(Q, epsilon)
            action = sample_action_from_policy(policy, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # target value to achieve.
            target = reward + gamma * (1 - done) * np.max(Q[next_state])
            # update the Q-table towards the target, based on learning rate alpha.
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state

            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q, epsilon


def q_learning_with_history(
    env,
    num_episodes=5000,
    alpha=0.1,
    gamma=0.9,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    max_steps_per_episode=50,
    seed=7,
    snapshot_episodes=None,
):
    """Run Q-learning while recording selected intermediate training snapshots.

    Precondition:
    `env` is compatible with the Q-learning loop, and `snapshot_episodes`
    either is `None` or contains positive episode numbers.

    What happens:
    1. Perform the same learning loop as `q_learning()`.
    2. Capture copies of the current Q-table and behavior policy at selected
       episode counts.
    3. Store each snapshot together with the episode number and epsilon value
       used at that point in training.

    Postcondition:
    Returns the final Q-table, the collected snapshot records, and the final
    epsilon value.
    """
    random.seed(seed)
    np.random.seed(seed)
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    Q = np.zeros((num_states, num_actions))
    snapshots = []

    if snapshot_episodes is None:
        snapshot_episodes = [1, 10, 100, 500, 1000, num_episodes]
    snapshot_set = set(snapshot_episodes)

    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        done = False

        for _ in range(max_steps_per_episode):
            policy = epsilon_greedy_policy_from_q(Q, epsilon)
            action = sample_action_from_policy(policy, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            target = reward + gamma * (1 - done) * np.max(Q[next_state])
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state

            if done:
                break

        if episode in snapshot_set:
            snapshots.append(
                {
                    "episode": episode,
                    "epsilon": epsilon,
                    "Q": Q.copy(),
                    "policy": policy.copy(),
                }
            )

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q, snapshots, epsilon


## Run The Learner

Create the environment, train the Q-table, and print a sequence of saved snapshots. The final policy is the epsilon-greedy policy induced by the learned action values and the decayed exploration rate.


In [3]:
env = GridWorldCaseStudyEnv()
Q, snapshots, final_epsilon = q_learning_with_history(env)
policy = epsilon_greedy_policy_from_q(Q, final_epsilon)

np.set_printoptions(precision=3, suppress=True)
print("Q-Learning Snapshots")
for snapshot in snapshots:
    print(
        f"\nEpisode {snapshot['episode']} "
        f"(epsilon={snapshot['epsilon']:.3f})"
    )
    print("Q")
    print(snapshot["Q"])
    print("max_a Q(s, a)")
    print(snapshot["Q"].max(axis=1).reshape(3, 3))
    print(f"Epsilon-greedy behavior policy (epsilon={snapshot['epsilon']:.3f})")
    print(format_policy(snapshot["policy"]))

print("Learned action values Q")
print(Q)
print(f"\nFinal epsilon-greedy policy extracted from the learned Q-table (epsilon={final_epsilon:.3f})")
print(format_policy(policy))

payload = {
    "algorithm": "q_learning",
    "alpha": 0.1,
    "gamma": 0.9,
    "epsilon_start": 1.0,
    "epsilon_decay": 0.995,
    "epsilon_min": 0.01,
    "epsilon_final": float(final_epsilon),
    "num_episodes": 5000,
    "max_steps_per_episode": 50,
    "seed": 7,
    "action_order": [ACTION_NAMES[i].upper() for i in range(env.action_space.n)],
    "final": {
        "policy_matrix": policy.tolist(),
        "Q": Q.tolist(),
        "state_scores": Q.max(axis=1).tolist(),
    },
}

output_path = Path("q_learning_policies.json")
output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"\nExported JSON: {output_path.resolve()}")


Q-Learning Snapshots

Episode 1 (epsilon=1.000)
Q
[[-0.19  -0.28  -0.271 -0.1  ]
 [-0.1   -0.1   -0.19  -0.199]
 [ 0.    -0.19  -0.1   -0.109]
 [-0.279 -0.1    0.    -0.19 ]
 [ 0.    -0.271 -0.19  -0.19 ]
 [ 0.    -0.19   0.    -0.1  ]
 [ 0.    -0.19  -0.1    0.   ]
 [-0.19   1.     0.    -0.1  ]
 [ 0.     0.     0.     0.   ]]
max_a Q(s, a)
[[-0.1 -0.1  0. ]
 [ 0.   0.   0. ]
 [ 0.   1.   0. ]]
Epsilon-greedy behavior policy (epsilon=1.000)
+---+---+---+
|^>v<|^>v<|^>v<|
+---+---+---+
|^>v<|^>v<|^>v<|
+---+---+---+
|^>v<|^>v<| G |
+---+---+---+

Episode 10 (epsilon=0.956)
Q
[[-1.14  -0.89  -0.892 -0.58 ]
 [-0.441 -0.265 -0.608 -0.703]
 [-0.192 -0.254  0.505 -0.215]
 [-0.838 -0.807 -0.44  -0.979]
 [-0.414 -0.207  0.058 -0.554]
 [-0.183 -0.003  3.211  1.566]
 [-0.518 -0.109 -0.384 -0.598]
 [-0.42   2.621  0.053 -0.384]
 [ 0.     0.     0.     0.   ]]
max_a Q(s, a)
[[-0.58  -0.265  0.505]
 [-0.44   0.058  3.211]
 [-0.109  2.621  0.   ]]
Epsilon-greedy behavior policy (epsilon=0.956)
+---